<a href="https://colab.research.google.com/github/BreakTheAlgorithm/MLforALL/blob/main/chapter_06_numpy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style='background:linear-gradient(135deg,#1B3A6B,#2D5AA0);padding:30px;border-radius:12px;color:white;font-family:Arial'>
<h1 style='margin:0;font-size:28px'>🔢 Chapter 6 — NumPy: The Numerical Backbone</h1>
<p style='margin:8px 0 0'>Book: Machine Learning — From Zero to Engineer | Est. Time: 90 mins | Level: Intermediate</p>
</div>

## 📋 Learning Objectives

By the end of this notebook, you will be able to:
- Create and inspect NumPy ndarrays (shape, dtype, indexing)
- Apply boolean masking, reshaping, and stacking operations
- Use vectorised operations and broadcasting efficiently
- Compute statistical summaries along axes
- Perform linear algebra operations (dot product, transpose, inverse)
- Implement ML formulas (MSE, normalisation, linear regression) from scratch

---

In [ ]:
# ─────────────────────────────────────────────────────────────
# Setup & Imports
# ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
np.random.seed(42)
print('✅ NumPy version:', np.__version__)
print('✅ Setup complete!')

## 📖 Section A — The ndarray

NumPy's `ndarray` is 50–100× faster than Python lists because it stores data in **contiguous memory** with a fixed type — no object overhead per element.

| Property | What it tells you |
|----------|------------------|
| `.shape` | Dimensions as tuple, e.g. `(3, 4)` |
| `.ndim`  | Number of axes — 1D, 2D, 3D… |
| `.size`  | Total number of elements |
| `.dtype` | Data type: `float64`, `int32`, `bool`… |

> 💡 **Key Rule**
> `arr.shape` → `(rows, cols)` for 2D arrays
> First index = rows, second index = columns — same convention as matrices in maths.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Demo: Creating and Inspecting ndarrays
# ─────────────────────────────────────────────────────────────
np.random.seed(42)

# Creation methods
a = np.array([1, 2, 3, 4, 5])
b = np.zeros((3, 4))
c = np.ones((2, 3))
d = np.eye(4)
e = np.arange(0, 10, 2)
f = np.linspace(0, 1, 5)
g = np.random.randint(1, 50, size=(3, 3))

print('np.array([1..5])        :', a, '| shape:', a.shape, '| dtype:', a.dtype)
print('np.zeros((3,4))  shape  :', b.shape)
print('np.eye(4):\n', d)
print('np.arange(0,10,2)       :', e)
print('np.linspace(0,1,5)      :', f)
print('np.random.randint(3×3):\n', g)
print(f'g → shape:{g.shape} ndim:{g.ndim} size:{g.size} dtype:{g.dtype}')

## 📖 Section B — Indexing and Slicing

```python
arr[row, col]          # single element
arr[row, :]            # entire row
arr[:, col]            # entire column
arr[r1:r2, c1:c2]      # submatrix
arr[arr > 50]          # boolean mask → 1D array of matching values
arr[arr > 50] = 0      # in-place assignment via mask
```

> 💡 **Key Rule**
> `arr[condition]` returns all elements where `condition` is `True`.
> `arr[condition] = value` modifies those elements in-place — no loop needed.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Demo: Indexing, Slicing, and Boolean Masking
# ─────────────────────────────────────────────────────────────
np.random.seed(42)
mat = np.random.randint(10, 90, size=(4, 5))
print('Matrix:\n', mat)

print('\nRow 2               :', mat[2, :])
print('Column 3            :', mat[:, 3])
print('Top-left 2×2        :\n', mat[:2, :2])
print('Elements > 70       :', mat[mat > 70])

mat_copy = mat.copy()
mat_copy[mat_copy > 70] = 0
print('\nAfter zeroing > 70:\n', mat_copy)

## 📖 Section C — Reshaping & Stacking

```python
arr.reshape(rows, cols)   # new shape — must have same total elements
arr.reshape(-1, 4)        # -1 means 'figure out this dimension'
arr.flatten()             # always returns a copy → 1D
arr.ravel()               # returns a view where possible → 1D (faster)
np.vstack([a, b])         # stack rows (vertical)
np.hstack([a, b])         # stack columns (horizontal)
```

> 💡 **Key Rule**
> `flatten()` → always a copy (safe to modify). `ravel()` → view when possible (faster, but modifying it may change original).

In [ ]:
# ─────────────────────────────────────────────────────────────
# Demo: Reshaping and Stacking
# ─────────────────────────────────────────────────────────────
arr = np.arange(1, 25)
print('Original 1D:', arr)
mat = arr.reshape(4, 6)
print('Reshaped to (4,6):\n', mat)
print('Reshape with -1:', arr.reshape(-1, 4).shape, '← 6 rows auto-computed')

a = np.array([[1, 2], [3, 4]])
b = np.array([[5, 6], [7, 8]])
print('\nvstack:\n', np.vstack([a, b]))
print('hstack:\n', np.hstack([a, b]))

## 📖 Section D — Vectorised Operations & Broadcasting

Broadcasting: NumPy can combine arrays of different shapes by **stretching** size-1 dimensions.

**Rule**: Compare shapes from the **right**. Dimensions must be equal OR one of them must be 1.

```
Shape (4, 3) + shape (3,) → broadcast (3,) to (4, 3) ✅
Shape (4, 3) + shape (4,) → ERROR ❌ (align from right: 3 ≠ 4)
Shape (4, 3) + shape (4,1) → broadcast ✅
```

> 💡 **Key Rule**
> To subtract column means from a matrix: `mat - mat.mean(axis=0)` — this subtracts each column's mean from every row. The `(3,)` mean broadcasts against `(4,3)`.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Demo: Broadcasting
# ─────────────────────────────────────────────────────────────
np.random.seed(42)
scores = np.random.randint(50, 100, size=(4, 3))  # 4 students, 3 exams
weights = np.array([0.3, 0.4, 0.3])               # weights per exam

print('Scores (4 students × 3 exams):\n', scores)
print('\nWeights:', weights, '← shape (3,) broadcasts against (4,3)')

weighted = scores * weights          # broadcasting: (4,3) * (3,) → (4,3)
final_scores = weighted.sum(axis=1)  # sum across columns → (4,)
print('\nWeighted scores per student:', np.round(final_scores, 2))

# Column-wise mean subtraction (z-score preprocessing)
col_means = scores.mean(axis=0)
col_stds  = scores.std(axis=0)
z_scores  = (scores - col_means) / col_stds
print('\nZ-normalised (column means ≈ 0):', np.round(z_scores.mean(axis=0), 10))

## 📖 Section E — Statistical Operations

```python
arr.mean(axis=0)  # mean of each column (collapse rows)
arr.mean(axis=1)  # mean of each row (collapse cols)
np.argmax(arr)    # index of maximum (flattened)
np.percentile(arr, 75)  # 75th percentile
arr.cumsum()      # cumulative sum
```

> 💡 **Key Mental Model**
> `axis=0` → **collapse rows** (you get one value per column)
> `axis=1` → **collapse columns** (you get one value per row)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Demo: Statistical Operations with axis
# ─────────────────────────────────────────────────────────────
np.random.seed(42)
data = np.random.randint(40, 100, size=(5, 4))
print('Data (5 rows = students, 4 cols = exams):\n', data)
print('\nMean per exam  (axis=0):', data.mean(axis=0).round(1))
print('Mean per student (axis=1):', data.mean(axis=1).round(1))
print('Std per exam   (axis=0):', data.std(axis=0).round(1))
print('Global max :', data.max(), '| at index:', np.unravel_index(data.argmax(), data.shape))
print('75th percentile:', np.percentile(data, 75))

## 📖 Section F — Linear Algebra

```python
np.dot(a, b)   # dot product (also: a @ b for matrices)
a.T            # transpose
np.linalg.inv(A)   # matrix inverse
np.linalg.det(A)   # determinant
np.linalg.norm(v)  # Euclidean length of vector
```

> 💡 **Key Rule**
> The `@` operator is matrix multiplication (Python 3.5+). Use it instead of `np.dot()` for readability.
> `W = np.linalg.inv(X.T @ X) @ X.T @ y` is the closed-form OLS linear regression solution.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Demo: Linear Algebra — the building blocks of ML
# ─────────────────────────────────────────────────────────────
weights   = np.array([0.4, 0.3, 0.3])     # model weights
features  = np.array([5.0, 8.0, 6.0])     # input features

# Dot product = prediction in linear model (w · x)
prediction = np.dot(weights, features)
print(f'Dot product (w·x): {prediction}')
print(f'Using @ operator : {weights @ features}')

A = np.array([[2, 1], [5, 3]], dtype=float)
print('\nMatrix A:\n', A)
print('Transpose A.T:\n', A.T)
print('Inverse A⁻¹:\n', np.linalg.inv(A).round(3))
print('A @ A⁻¹ (should be I):\n', (A @ np.linalg.inv(A)).round(6))
print('\nNorm of features vector:', np.linalg.norm(features).round(4))

---
## 🟢 Exercises 6.1–6.3  |  Estimated: 20 min
---

> 🎯 **Exercise 6.1 — Array Creation and Inspection** *(Level 1 · Est. 5–8 min)*
>
> Create a 5×5 matrix of random integers between 1 and 50 (seed=42).
> Print shape, size, dtype, element at [3,2].
> Replace all values > 40 with 0.
>
> **Expected:** `matrix.shape == (5,5)`, `matrix.max() <= 40`

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 6.1: Array Creation and Inspection
# Estimated time: 5-8 minutes
# ─────────────────────────────────────────────────────────────
np.random.seed(42)

# Create a 5×5 matrix of random integers between 1 and 50
matrix = # YOUR CODE HERE

# Print shape, size, dtype, element at [3,2]
# YOUR CODE HERE

# Replace all values > 40 with 0
# YOUR CODE HERE

assert matrix.shape == (5, 5),      f'Expected (5,5), got {matrix.shape}'
assert matrix.max() <= 40,          f'Max value should be ≤ 40 after zeroing, got {matrix.max()}'
print('\n✅ Array creation and masking verified!')

In [ ]:
# @title ✅ Solution — Exercise 6.1 (click to expand)
# @markdown > Run this cell to reveal the solution
np.random.seed(42)
matrix = np.random.randint(1, 51, size=(5, 5))   # 51 because high is exclusive
print('Original matrix:\n', matrix)
print(f'shape={matrix.shape} size={matrix.size} dtype={matrix.dtype} [3,2]={matrix[3,2]}')

matrix[matrix > 40] = 0   # boolean mask in-place assignment
print('\nAfter zeroing values > 40:\n', matrix)

assert matrix.shape == (5, 5)
assert matrix.max() <= 40
print('\n✅ Solution verified!')

> 🎯 **Exercise 6.2 — Slicing Practice** *(Level 1 · Est. 5–8 min)*
>
> Create a 6×6 matrix using `np.arange(1, 37).reshape(6,6)`.
> Extract: a) last two rows, b) first three columns, c) 3×3 submatrix from rows 1-3 cols 1-3, d) diagonal

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 6.2: Slicing Practice
# Estimated time: 5-8 minutes
# ─────────────────────────────────────────────────────────────

mat = np.arange(1, 37).reshape(6, 6)
print('Matrix:\n', mat)

# a) Last two rows
last_two_rows   = # YOUR CODE HERE
print('\na) Last two rows:\n', last_two_rows)

# b) First three columns
first_three_cols = # YOUR CODE HERE
print('\nb) First three columns:\n', first_three_cols)

# c) 3×3 submatrix from rows 1-3, cols 1-3
submatrix       = # YOUR CODE HERE
print('\nc) 3×3 submatrix (rows 1-3, cols 1-3):\n', submatrix)

# d) Diagonal elements
diagonal        = # YOUR CODE HERE
print('\nd) Diagonal:', diagonal)

assert last_two_rows.shape  == (2, 6), f'Expected (2,6) got {last_two_rows.shape}'
assert first_three_cols.shape == (6, 3)
assert submatrix.shape      == (3, 3)
assert diagonal.shape       == (6,)
print('\n✅ All slices correct!')

In [ ]:
# @title ✅ Solution — Exercise 6.2 (click to expand)
# @markdown > Run this cell to reveal the solution
mat = np.arange(1, 37).reshape(6, 6)

last_two_rows    = mat[-2:, :]        # negative indexing counts from end
first_three_cols = mat[:, :3]         # all rows, first 3 columns
submatrix        = mat[1:4, 1:4]      # rows 1,2,3 and cols 1,2,3
diagonal         = np.diag(mat)       # np.diag() on 2D array returns main diagonal

print('Last two rows:\n', last_two_rows)
print('First three cols:\n', first_three_cols)
print('3×3 Submatrix:\n', submatrix)
print('Diagonal:', diagonal)

assert last_two_rows.shape  == (2, 6)
assert first_three_cols.shape == (6, 3)
assert submatrix.shape      == (3, 3)
assert diagonal.shape       == (6,)
print('\n✅ Solution verified!')

> 🎯 **Exercise 6.3 — Vectorised Salary Operations** *(Level 1 · Est. 8 min)*
>
> ```python
> salaries = np.array([320000, 450000, 600000, 800000, 1200000])
> ```
> a) Apply 10% hike (one line, no loop); b) Boolean mask: which salaries exceed 7 LPA after hike?; c) Min-Max normalise to [0,1]

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 6.3: Vectorised Salary Operations
# Estimated time: 8 minutes
# ─────────────────────────────────────────────────────────────

salaries = np.array([320000, 450000, 600000, 800000, 1200000])

# a) Apply 10% hike (one line, no loop)
hiked = # YOUR CODE HERE
print('a) After 10% hike:', hiked)

# b) Which salaries exceed 7 LPA (700,000) after hike?
above_7lpa = # YOUR CODE HERE
print('b) Above 7 LPA:', above_7lpa)

# c) Min-Max normalise hiked salaries to [0, 1]
normalised = # YOUR CODE HERE
print('c) Normalised:', normalised.round(4))

assert abs(hiked[0] - 352000) < 1,         'First hiked salary should be 352000'
assert normalised.min() == 0.0,             'Min of normalised should be 0.0'
assert abs(normalised.max() - 1.0) < 1e-9, 'Max of normalised should be 1.0'
print('\n✅ Vectorised operations verified!')

In [ ]:
# @title ✅ Solution — Exercise 6.3 (click to expand)
# @markdown > Run this cell to reveal the solution

salaries = np.array([320000, 450000, 600000, 800000, 1200000])

# a) Multiply entire array by 1.10 — no loop, C-speed under the hood
hiked = (salaries * 1.10).astype(int)
print('a) After 10% hike:', hiked)

# b) Boolean indexing — returns elements where condition is True
above_7lpa = hiked[hiked > 700000]
print('b) Above 7 LPA:', above_7lpa)

# c) Min-Max formula: (x - min) / (max - min)
normalised = (hiked - hiked.min()) / (hiked.max() - hiked.min())
print('c) Normalised:', normalised.round(4))

assert abs(hiked[0] - 352000) < 1
assert normalised.min() == 0.0
assert abs(normalised.max() - 1.0) < 1e-9
print('\n✅ This is exactly how sklearn\'s MinMaxScaler works internally!')

---
## 🟡 Exercises 6.4–6.7  |  Estimated: 35 min
---

> 🎯 **Exercise 6.4 — Boolean Masking & Filtering** *(Level 2 · Est. 10 min)*
>
> `np.random.seed(42); scores = np.random.randint(0, 100, size=(50,))`
> a) Count students who scored above 80; b) Average score of failed students (< 40); c) Floor-clip scores below 40 to exactly 40; d) Rank all students

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 6.4: Boolean Masking and Filtering
# Estimated time: 10 minutes
# ─────────────────────────────────────────────────────────────
np.random.seed(42)
scores = np.random.randint(0, 100, size=(50,))

# a) Count students who scored above 80
above_80_count = # YOUR CODE HERE
print(f'a) Students above 80: {above_80_count}')

# b) Average score of students who failed (< 40)
fail_avg = # YOUR CODE HERE
print(f'b) Average of failed students: {fail_avg:.2f}')

# c) Floor-clip: replace scores below 40 with exactly 40
scores_clipped = scores.copy()
# YOUR CODE HERE
print(f'c) Min score after clip: {scores_clipped.min()} (should be 40)')

# d) Rank all students (argsort of argsort = ranks)
ranks = # YOUR CODE HERE
print(f'd) Rank of first student: {ranks[0]} (out of 50)')

assert scores_clipped.min() == 40, 'Min should be 40 after clipping'
assert len(ranks) == 50, 'Must have a rank for all 50 students'
print('\n✅ Boolean masking exercises verified!')

In [ ]:
# @title ✅ Solution — Exercise 6.4 (click to expand)
# @markdown > Run this cell to reveal the solution
np.random.seed(42)
scores = np.random.randint(0, 100, size=(50,))

# a) Count using boolean mask — True is treated as 1 in sum()
above_80_count = (scores > 80).sum()
print(f'a) Students above 80: {above_80_count}')

# b) Mask to select only failing scores, then take mean
fail_avg = scores[scores < 40].mean()
print(f'b) Average of failed students: {fail_avg:.2f}')

# c) In-place boolean assignment
scores_clipped = scores.copy()
scores_clipped[scores_clipped < 40] = 40
print(f'c) Min after clip: {scores_clipped.min()}')

# d) argsort gives indices that would sort the array
#    argsort of argsort gives the rank of each element
ranks = np.argsort(np.argsort(scores)) + 1   # +1 for 1-based ranks
print(f'd) Ranks (first 5): {ranks[:5]}')

assert scores_clipped.min() == 40
assert len(ranks) == 50
print('\n✅ Solution verified! Note: argsort(argsort(x)) is a neat NumPy trick for ranking.')

> 🎯 **Exercise 6.5 — Reshaping for ML (MNIST-style)** *(Level 2 · Est. 10 min)*
>
> Work with flat image vectors as used in deep learning:
> a) Reshape `flat_image` (784,) → (28, 28); b) Normalise [0,255] → [0.0,1.0]; c) Flatten back; d) Stack 5 images → shape (5, 784)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 6.5: Reshaping for ML (MNIST-style image data)
# Estimated time: 10 minutes
# ─────────────────────────────────────────────────────────────
np.random.seed(0)
flat_image = np.random.randint(0, 256, 784)   # simulated 28×28 greyscale image

# a) Reshape to (28, 28)
image_matrix = # YOUR CODE HERE
print(f'a) image_matrix.shape = {image_matrix.shape}')

# b) Normalise pixel values from [0,255] to [0.0, 1.0]
normalised = # YOUR CODE HERE
print(f'b) normalised max={normalised.max():.3f} min={normalised.min():.3f}')

# c) Reshape back to flat (784,)
flat_again = # YOUR CODE HERE
print(f'c) flat_again.shape = {flat_again.shape}')

# d) Stack 5 such images — final shape should be (5, 784)
all_images = np.vstack([np.random.randint(0, 256, 784).reshape(1, -1) for _ in range(5)])
print(f'd) all_images.shape = {all_images.shape}')

assert image_matrix.shape == (28, 28),    f'Expected (28,28), got {image_matrix.shape}'
assert normalised.max()   <= 1.0,         'Max of normalised should be ≤ 1.0'
assert normalised.min()   >= 0.0,         'Min of normalised should be ≥ 0.0'
assert all_images.shape   == (5, 784),    f'Expected (5,784), got {all_images.shape}'
print('\n✅ Image reshaping verified — same pattern as loading MNIST in real deep learning!')

In [ ]:
# @title ✅ Solution — Exercise 6.5 (click to expand)
# @markdown > Run this cell to reveal the solution
np.random.seed(0)
flat_image = np.random.randint(0, 256, 784)

image_matrix = flat_image.reshape(28, 28)      # reshape to 2D grid
normalised   = image_matrix / 255.0            # divide by max pixel value
flat_again   = normalised.flatten()            # flatten returns a 1D copy

# Stack 5 images — each reshaped to (1, 784) then vstack combines them
np.random.seed(0)
all_images = np.vstack([np.random.randint(0, 256, 784).reshape(1, -1) for _ in range(5)])

print(f'image_matrix: {image_matrix.shape}')
print(f'normalised:   max={normalised.max():.3f} min={normalised.min():.3f}')
print(f'flat_again:   {flat_again.shape}')
print(f'all_images:   {all_images.shape}')

assert image_matrix.shape == (28, 28)
assert normalised.max() <= 1.0
assert all_images.shape == (5, 784)
print('\n✅ Solution verified! In real MNIST pipelines X.shape is (60000, 784) — same pattern, bigger numbers.')

> 🎯 **Exercise 6.7 — ML Formulas from Scratch** *(Level 2 · Est. 15 min)*
>
> Implement using ONLY NumPy (no loops, no sklearn):
> a) MSE between actual and predicted; b) Euclidean distance between two points; c) Dot product; d) Sigmoid function

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 6.7: ML Formulas from Scratch
# Estimated time: 15 minutes
# ─────────────────────────────────────────────────────────────

actual    = np.array([3.0, 5.0, 2.0, 7.0, 9.0])
predicted = np.array([2.5, 5.2, 1.8, 7.5, 8.0])
p1 = np.array([1.0, 2.0, 3.0])
p2 = np.array([4.0, 6.0, 3.0])
w  = np.array([0.4, 0.3, 0.3])
f  = np.array([5.0, 8.0, 6.0])
z  = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])

# a) Mean Squared Error
mse = # YOUR CODE HERE
print(f'a) MSE: {mse:.4f}  (expected ≈ 0.1460)')

# b) Euclidean distance between p1 and p2
distance = # YOUR CODE HERE
print(f'b) Euclidean distance: {distance:.4f}  (expected ≈ 5.0000)')

# c) Dot product of w and f
dot = # YOUR CODE HERE
print(f'c) Dot product: {dot:.4f}  (expected ≈ 5.0000)')

# d) Sigmoid: 1 / (1 + e^(-z))
sigmoid = # YOUR CODE HERE
print(f'd) Sigmoid: {sigmoid.round(4)}')

assert abs(mse      - 0.146) < 0.001, f'MSE wrong: {mse}'
assert abs(distance - 5.0)   < 0.001, f'Distance wrong: {distance}'
assert abs(dot      - 5.0)   < 0.001, f'Dot product wrong: {dot}'
assert sigmoid[2] == 0.5,             'Sigmoid(0) should be exactly 0.5'
print('\n✅ All ML formula implementations correct!')

In [ ]:
# @title ✅ Solution — Exercise 6.7 (click to expand)
# @markdown > Run this cell to reveal the solution

actual    = np.array([3.0, 5.0, 2.0, 7.0, 9.0])
predicted = np.array([2.5, 5.2, 1.8, 7.5, 8.0])
p1 = np.array([1.0, 2.0, 3.0])
p2 = np.array([4.0, 6.0, 3.0])
w  = np.array([0.4, 0.3, 0.3])
f  = np.array([5.0, 8.0, 6.0])
z  = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])

# a) MSE = mean of squared differences
mse = np.mean((actual - predicted) ** 2)
print(f'a) MSE: {mse:.4f}')

# b) Euclidean = sqrt(sum of squared differences) — same as np.linalg.norm
distance = np.sqrt(np.sum((p2 - p1) ** 2))
print(f'b) Distance: {distance:.4f}')

# c) np.dot or @ — both give scalar for 1D arrays
dot = np.dot(w, f)
print(f'c) Dot product: {dot:.4f}')

# d) Sigmoid = 1 / (1 + e^(-z))  — np.exp handles arrays element-wise
sigmoid = 1 / (1 + np.exp(-z))
print(f'd) Sigmoid: {sigmoid.round(4)}')

assert abs(mse      - 0.146) < 0.001
assert abs(distance - 5.0)   < 0.001
assert abs(dot      - 5.0)   < 0.001
assert sigmoid[2]   == 0.5
print('\n✅ These 4 formulas — MSE, distance, dot product, sigmoid — are the building blocks of almost all ML algorithms.')

---
## 🔴 Exercises 6.8–6.10  |  Estimated: 45 min
---

> 🎯 **Exercise 6.8 — Linear Regression from Scratch** *(Level 3 · Est. 20 min)*
>
> Using closed-form formula, compute slope `m` and intercept `b`, predict for X=6, compute MSE.
> Formula: `m = (n·Σxy - Σx·Σy) / (n·Σx² - (Σx)²)`

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 6.8: Linear Regression from Scratch
# Estimated time: 20-25 minutes
# ─────────────────────────────────────────────────────────────

X = np.array([1, 2, 3, 4, 5], dtype=float)
y = np.array([3, 5, 4, 6, 7], dtype=float)

# a) Compute slope m and intercept b using the closed-form formula
n = len(X)
m = # YOUR CODE HERE
b = # YOUR CODE HERE
print(f'a) Slope (m): {m:.4f}  Intercept (b): {b:.4f}')

# b) Predict for X=6
pred_6 = # YOUR CODE HERE
print(f'b) Prediction for X=6: {pred_6:.4f}')

# c) Compute MSE on training data
y_hat = # YOUR CODE HERE  (predictions for all training X)
mse   = # YOUR CODE HERE
print(f'c) Training MSE: {mse:.4f}  (expected < 1.0, solution ≈ 0.24)')

assert abs(m - 0.9) < 0.1,    f'Slope looks wrong: {m:.4f}'
assert mse < 1.0,              f'MSE should be < 1.0, got {mse:.4f}'
print('\n✅ Linear regression from scratch — the same maths sklearn uses!')

In [ ]:
# @title ✅ Solution — Exercise 6.8 (click to expand)
# @markdown > Run this cell to reveal the solution

X = np.array([1, 2, 3, 4, 5], dtype=float)
y = np.array([3, 5, 4, 6, 7], dtype=float)
n = len(X)

# Closed-form slope formula — directly from the OLS derivation
m = (n * np.sum(X * y) - np.sum(X) * np.sum(y)) / \
    (n * np.sum(X**2) - np.sum(X)**2)

# Intercept from the mean-based formula
b = np.mean(y) - m * np.mean(X)

print(f'Slope: {m:.4f}  Intercept: {b:.4f}')
print(f'Equation: y = {m:.4f}x + {b:.4f}')

pred_6 = m * 6 + b
print(f'Prediction for X=6: {pred_6:.4f}')

y_hat = m * X + b                    # vectorised prediction for all training points
mse   = np.mean((y - y_hat) ** 2)
print(f'Training MSE: {mse:.4f}')

assert abs(m - 0.9) < 0.1
assert mse < 1.0
print('\n✅ Congratulations — you just implemented the oldest machine learning algorithm in existence!')
print('   sklearn.LinearRegression does the same thing, but for n-dimensional features using matrix algebra.')

> 🎯 **Exercise 6.9 — Cosine Similarity** *(Level 3 · Est. 15 min)*
>
> Compute cosine similarity between two document vectors, then find the most similar document from a collection.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 6.9: Cosine Similarity
# Estimated time: 15 minutes
# ─────────────────────────────────────────────────────────────

doc1 = np.array([1, 1, 0, 1, 0, 1], dtype=float)
doc2 = np.array([1, 0, 1, 1, 0, 0], dtype=float)

# a) Compute cosine similarity between doc1 and doc2
# Formula: cos_sim = (a · b) / (||a|| × ||b||)
cosine_sim = # YOUR CODE HERE
print(f'a) Cosine similarity(doc1, doc2): {cosine_sim:.4f}')

# b) Find the most similar document to doc1 from a collection of 5 random binary vectors
np.random.seed(7)
collection = np.random.randint(0, 2, size=(5, 6)).astype(float)

def most_similar(query, collection):
    """Return the index of the most similar document to query."""
    # YOUR CODE HERE
    pass

best_idx = most_similar(doc1, collection)
print(f'b) Most similar document index: {best_idx}')

assert 0 <= best_idx < 5, 'Index must be in range 0-4'
print('\n✅ Cosine similarity implemented!')

In [ ]:
# @title ✅ Solution — Exercise 6.9 (click to expand)
# @markdown > Run this cell to reveal the solution

doc1 = np.array([1, 1, 0, 1, 0, 1], dtype=float)
doc2 = np.array([1, 0, 1, 1, 0, 0], dtype=float)

# a) Cosine similarity — dot product divided by product of norms
cosine_sim = np.dot(doc1, doc2) / (np.linalg.norm(doc1) * np.linalg.norm(doc2))
print(f'a) Cosine similarity: {cosine_sim:.4f}')
print('   Interpretation: 1.0 = identical direction, 0.0 = orthogonal, -1.0 = opposite')

# b) Compute similarity to each document in collection, return argmax
np.random.seed(7)
collection = np.random.randint(0, 2, size=(5, 6)).astype(float)

def most_similar(query, collection):
    similarities = []
    for doc in collection:
        norm_product = np.linalg.norm(query) * np.linalg.norm(doc)
        if norm_product == 0:
            sim = 0.0  # handle zero vector edge case
        else:
            sim = np.dot(query, doc) / norm_product
        similarities.append(sim)
    return np.argmax(similarities)

best_idx = most_similar(doc1, collection)
print(f'b) Most similar document: index {best_idx}')

assert 0 <= best_idx < 5
print('\n✅ Cosine similarity powers search engines, recommendation systems, and NLP embedding comparisons!')

---
## 🎤 Interview Q&A

<details>
<summary><strong>Q1: NumPy array vs Python list — when would you use each?</strong></summary>

**Answer:** Use NumPy for numerical computation — it's 50-100× faster due to contiguous memory and C-optimised operations. Use Python lists when you need mixed types, dynamic appending, or non-numerical data. In ML: all data matrices are NumPy arrays; configurations, model names, and metadata stay as Python lists/dicts.
</details>

<details>
<summary><strong>Q2: What is the difference between axis=0 and axis=1?</strong></summary>

**Answer:** `axis=0` collapses across rows — you get one result per column. `axis=1` collapses across columns — you get one result per row. Memory aid: `axis=0` → the 0th dimension (rows) disappears. For a (4, 3) matrix: `.mean(axis=0)` → shape (3,), `.mean(axis=1)` → shape (4,).
</details>

<details>
<summary><strong>Q3: Explain broadcasting with a concrete example.</strong></summary>

**Answer:** Broadcasting lets NumPy perform operations on arrays with different shapes by stretching size-1 dimensions. Example: subtracting column means from a `(100, 5)` dataset. `mat.mean(axis=0)` returns shape `(5,)`. `mat - mat.mean(axis=0)` broadcasts the `(5,)` vector across all 100 rows — equivalent to writing a Python loop 100 times, but done in C-speed in one line.
</details>

<details>
<summary><strong>Q4: flatten() vs ravel() — what's the practical difference?</strong></summary>

**Answer:** Both return a 1D version of the array. `flatten()` always returns a **copy** — safe to modify without affecting the original. `ravel()` returns a **view** when possible — faster but changes propagate back to the original. Use `flatten()` when you need to safely modify the result; use `ravel()` when you only need to read it.
</details>

<details>
<summary><strong>Q5: What is np.argmax() and where is it used in ML?</strong></summary>

**Answer:** `np.argmax(arr)` returns the **index** of the maximum value, not the value itself. In ML classification: the model's output layer produces probabilities for each class — `np.argmax(probs)` gives the predicted class index. Example: `[0.1, 0.7, 0.2]` → `argmax` → `1` (class 1 predicted).
</details>

---

<div style='background:#EDF7F0;border-left:6px solid #2E7D50;padding:20px;border-radius:8px;margin-top:20px'>
<h3 style='color:#2E7D50'>✅ Chapter 6 Complete!</h3>
<p>Impressive work — you've covered the entire NumPy toolkit used in production ML pipelines:</p>
<ul>
<li>Array creation, inspection, and boolean masking</li>
<li>Reshaping, stacking, and the -1 trick</li>
<li>Vectorised operations and broadcasting (the core of fast ML)</li>
<li>Statistical operations with axis — understanding axis=0 vs axis=1</li>
<li>Linear algebra: dot products, transposes, and matrix inverses</li>
<li>ML formulas from scratch: MSE, Euclidean distance, sigmoid, and linear regression</li>
</ul>
<p><strong>Next:</strong> Chapter 7 — Pandas: Data Wrangling for the Real World | <a href='#'>Open Chapter 7 Notebook →</a></p>
</div>